# StyleGAN3 — Alias-Free Synthesis

This notebook implements a simplified **StyleGAN3** (Karras et al., 2021), following Module 20.

StyleGAN2 cleaned up StyleGAN's AdaIN artifacts and added path length regularization. But it still operated on a *discrete pixel grid*, which ties features to specific spatial locations in ways that produce aliasing and unwanted texture-stickiness when the image is translated or rotated.

StyleGAN3 reframes synthesis as a **signal-processing** problem. Two architectural moves make the difference:

1. **Fourier features replace the learned constant input.** The synthesis network no longer starts from a learned `(1, C, 4, 4)` tensor. Instead it starts from a deterministic 2D Fourier feature map computed from continuous spatial coordinates. Coordinates become the *first-class input*, and the model is forced to treat them as continuous.
2. **Low-pass filters after each nonlinearity.** ReLU / LeakyReLU can create high-frequency content even when the input was appropriately band-limited. StyleGAN3 applies a Gaussian blur after every nonlinear layer to keep the frequency content under control — this is what kills the aliasing.

The architecture otherwise keeps StyleGAN2's recipe: mapping network `z -> w`, modulated convolutions with demodulation, and path length regularization.

## Implementation Plan

- **Fourier features**: deterministic 2D Fourier feature map at the base resolution, computed from a normalized `[-1, 1]` coordinate grid. Sampled once, frozen.
- **Filtered operations**: after every upsampling, apply a fixed Gaussian blur before the next modulated conv. This is the low-pass filter that prevents aliasing.
- **Modulated conv**: same `ModulatedConv2d` as StyleGAN2 — `F.unfold` + batched matmul for per-sample modulated weights, then demodulation.
- **Mapping network**: same as StyleGAN2.
- **Discriminator**: standard PatchGAN.
- **Loss**: adversarial `BCEWithLogitsLoss` + path length regularization (every `pl_every` iterations).
- **Equivariance demo**: a cell at the end that runs the Generator with three coordinate grids — default, translated, rotated — to show that the spatial structure of the output follows the coordinate input rather than being tied to a fixed pixel grid.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
import numpy as np
import os

torch.manual_seed(42)
np.random.seed(42)

## 1. Setup and Hyperparameters

In [ ]:
latent_dim   = 100
w_dim        = 128
img_channels = 1
img_size     = 32
features     = 64
batch_size   = 16
lr           = 2e-4
betas        = (0.5, 0.999)
epochs       = 25
pl_weight    = 1.0
pl_every     = 4
num_freqs    = 32

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

os.makedirs('samples_StyleGAN3', exist_ok=True)

## 2. Data — MNIST at 32x32

In [ ]:
transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

dataloader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=True, download=True, transform=transform),
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)

## 3. Weight Initialization

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)
    elif classname.find('Linear') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)

## 4. Fourier Features — Continuous Coordinate Encoding

Fourier features turn a `(H, W)` coordinate grid into a feature map of shape `(2 * num_freqs, H, W)`:

```
freq_i = random 2D frequency vector (frozen)
proj_i(x, y) = 2 * pi * (x * freq_i.x + y * freq_i.y)
feature_i   = (sin(proj_i), cos(proj_i))
```

These features are deterministic functions of the coordinates. If we *shift* the coordinate grid, the Fourier features shift too — the model has no choice but to treat them as continuous.

This replaces StyleGAN2's *learned* constant input. StyleGAN2's constant was an opaque `(1, C, 4, 4)` tensor; StyleGAN3's Fourier features are an explicit function of space.

In [ ]:
class FilteredUpsample(nn.Module):
    """Nearest-neighbor 2x upsample + fixed 3x3 Gaussian blur."""
    def __init__(self):
        super().__init__()
        k = torch.tensor([
            [1., 2., 1.],
            [2., 4., 2.],
            [1., 2., 1.],
        ]) / 16.0
        self.register_buffer('kernel', k.view(1, 1, 3, 3))

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        C = x.size(1)
        kernel = self.kernel.expand(C, 1, 3, 3).to(x.dtype)
        x = F.conv2d(x, kernel, padding=1, groups=C)
        return x


class Blur(nn.Module):
    """Fixed 3x3 Gaussian blur, no spatial change (used after nonlinear ops)."""
    def __init__(self):
        super().__init__()
        k = torch.tensor([
            [1., 2., 1.],
            [2., 4., 2.],
            [1., 2., 1.],
        ]) / 16.0
        self.register_buffer('kernel', k.view(1, 1, 3, 3))

    def forward(self, x):
        C = x.size(1)
        kernel = self.kernel.expand(C, 1, 3, 3).to(x.dtype)
        return F.conv2d(x, kernel, padding=1, groups=C)

## 5. Filtered Upsample — Low-Pass After Each Resolution Jump

Upsampling introduces high-frequency content. Without a low-pass filter immediately afterwards, that high-frequency content can alias once the next convolution samples it.

We do `nearest-neighbor 2x` upsampling then a fixed 3x3 Gaussian blur. The blur is *not learned* — it's a known good low-pass filter for this purpose.

In [ ]:
# (removed: duplicate of FilteredUpsample from earlier edit; the class is already defined in the previous cell together with Blur)

In [ ]:
# (removed: orphan fragment from earlier edit)

In [ ]:
class ModulatedConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, w_dim=128):
        super().__init__()
        self.in_ch = in_ch
        self.out_ch = out_ch
        self.kernel_size = kernel_size
        self.weight = nn.Parameter(
            torch.randn(out_ch, in_ch, kernel_size, kernel_size) * 0.02
        )
        self.bias = nn.Parameter(torch.zeros(out_ch))
        self.style_fc = nn.Linear(w_dim, in_ch)

    def forward(self, x, w):
        B = x.size(0)
        k = self.kernel_size
        H, W = x.size(2), x.size(3)
        style = self.style_fc(w)
        weight = self.weight.unsqueeze(0) * style.unsqueeze(1).unsqueeze(3).unsqueeze(4)
        weight_norm_sq = weight.pow(2).sum(dim=[2, 3, 4])
        demod = torch.rsqrt(weight_norm_sq + 1e-8)
        weight = weight * demod.unsqueeze(2).unsqueeze(3).unsqueeze(4)
        x_unfold = F.unfold(x, kernel_size=k, padding=k // 2)
        weight_flat = weight.view(B, self.out_ch, self.in_ch * k * k)
        out = weight_flat @ x_unfold
        out = out.view(B, self.out_ch, H, W)
        return out + self.bias.view(1, -1, 1, 1)

## 7. Mapping Network `z -> w`

Same as StyleGAN2.

In [ ]:
class MappingNetwork(nn.Module):
    def __init__(self, latent_dim=100, w_dim=128, n_layers=4):
        super().__init__()
        layers = []
        in_dim = latent_dim
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, w_dim))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            in_dim = w_dim
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)

## 8. The Synthesis Network — Fourier Features + Filtered Upsampling

Spatial flow:

```
fourier features (2*num_freqs, 4, 4)
  -> 1x1 conv project -> channels[0] @ 4x4
  -> mod-conv 4x4  -> LeakyReLU -> filter (blur)
  -> FilteredUpsample -> 8x8
  -> mod-conv 8x8  -> LeakyReLU -> filter (blur)
  -> FilteredUpsample -> 16x16
  -> mod-conv 16x16 -> LeakyReLU -> filter (blur)
  -> FilteredUpsample -> 32x32
  -> mod-conv 32x32 -> LeakyReLU -> filter (blur)
  -> to_rgb (1x1 conv) -> 32x32 image
```

Note: the blur is the *alias-control step*. After every nonlinear operation that could create high-frequency content, we apply the same fixed Gaussian blur to push the frequency content back below the Nyquist limit.

In [ ]:
        # Subsequent blocks: FilteredUpsample then mod-conv then Blur (no upsample)
        self.upsamples = nn.ModuleList()
        self.convs    = nn.ModuleList()
        self.blurs    = nn.ModuleList()
        for i in range(1, len(channels)):
            self.upsamples.append(FilteredUpsample())
            self.convs.append(
                ModulatedConv2d(channels[i-1], channels[i], 3, w_dim)
            )
            self.blurs.append(Blur())   # post-conv low-pass, no spatial change

        self.to_rgb = nn.Conv2d(channels[-1], img_channels, 1)

    def forward(self, w):
        # Fourier features at the base resolution
        feats = fourier_features(w.size(0), 4, 4)
        x = self.input_proj(feats)                    # 4x4, channels[0]
        x = self.conv_4(x, w)
        x = F.leaky_relu(x, 0.2, inplace=True)

        for upsample, conv, blur in zip(self.upsamples, self.convs, self.blurs):
            x = upsample(x)                           # 2x with low-pass
            x = conv(x, w)                            # mod-conv
            x = F.leaky_relu(x, 0.2, inplace=True)
            x = blur(x)                               # post-conv low-pass (no upsample)

        return self.to_rgb(x)

## 9. The Generator and Discriminator

The Generator composes the mapping network and the synthesis network. The Discriminator is the same PatchGAN as before.

In [ ]:
class StyleGAN3Generator(nn.Module):
    def __init__(self, latent_dim=100, w_dim=128, channels=(128, 64, 32, 16)):
        super().__init__()
        self.mapping   = MappingNetwork(latent_dim, w_dim)
        self.synthesis = SynthesisNetwork(w_dim, channels)

    def forward(self, z, return_w=False):
        w = self.mapping(z)
        img = self.synthesis(w)
        if return_w:
            return img, w
        return img


class PatchDiscriminator(nn.Module):
    def __init__(self, img_channels=1, features=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(img_channels, features, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(features, features * 2, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(features * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(features * 2, features * 4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(features * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(features * 4, features * 8, 3, 1, 1, bias=False),
            nn.InstanceNorm2d(features * 8),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(features * 8, 1, 3, 1, 1, bias=False),
        )

    def forward(self, x):
        return self.net(x)

## 10. Models, Optimizers, Loss

In [ ]:
G = StyleGAN3Generator(latent_dim=latent_dim, w_dim=w_dim).to(device)
D = PatchDiscriminator(img_channels=img_channels, features=features).to(device)

G.apply(weights_init)
D.apply(weights_init)

optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=betas)
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=betas)

bce = nn.BCEWithLogitsLoss()
a_running = torch.tensor(0.0, device=device)

print(f'G params: {sum(p.numel() for p in G.parameters()):,}')
print(f'D params: {sum(p.numel() for p in D.parameters()):,}')

## 11. Fixed Noise for Visualization

In [ ]:
fixed_z = torch.randn(64, latent_dim, device=device)

## 12. Path Length Regularization Step

Same trick as StyleGAN2 — fresh batch, compute `<fake, y>.sum().backward()` wrt `w`, take the norm, penalize `(norm - a_running)^2`. Run every `pl_every` iterations.

In [ ]:
def path_length_step(G, optimizer_G, pl_weight):
    global a_running
    optimizer_G.zero_grad()
    z_pl = torch.randn(batch_size, latent_dim, device=device)
    w_pl = G.mapping(z_pl)
    fake_pl = G.synthesis(w_pl)
    y_pl = torch.randn_like(fake_pl)

    grads = torch.autograd.grad(
        outputs=(fake_pl * y_pl).sum(),
        inputs=[w_pl],
        create_graph=True,
        retain_graph=True,
    )[0]
    pl = grads.pow(2).sum(dim=1).sqrt().mean()
    pl_penalty = (pl - a_running) ** 2
    (pl_weight * pl_penalty).backward()
    optimizer_G.step()

    a_running = a_running + 0.01 * (pl.detach() - a_running)
    return pl.item(), pl_penalty.item()

## 13. The Training Loop

In [ ]:
losses_g = []
losses_d = []
pl_history = []

G.train(); D.train()
step = 0

for epoch in range(epochs):
    sum_d = 0.0
    sum_g = 0.0
    n_batches = 0

    for real_imgs, _ in dataloader:
        real_imgs = real_imgs.to(device)
        b = real_imgs.size(0)

        valid  = torch.ones (b, 1, 4, 4, device=device)
        fake_t = torch.zeros(b, 1, 4, 4, device=device)

        # -----------------
        # Phase A: Train D
        # -----------------
        optimizer_D.zero_grad()
        real_logits = D(real_imgs)
        d_real_loss = bce(real_logits, valid)

        z = torch.randn(b, latent_dim, device=device)
        with torch.no_grad():
            gen_imgs = G(z)
        fake_logits = D(gen_imgs)
        d_fake_loss = bce(fake_logits, fake_t)

        d_loss = (d_real_loss + d_fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()

        # -----------------
        # Phase B: Train G
        # -----------------
        optimizer_G.zero_grad()
        gen_imgs = G(z)
        validity_logits = D(gen_imgs)
        g_loss = bce(validity_logits, valid)
        g_loss.backward()
        optimizer_G.step()

        # -----------------
        # Phase C (optional): PLR
        # -----------------
        if step % pl_every == 0:
            pl, pl_pen = path_length_step(G, optimizer_G, pl_weight)
            pl_history.append(pl)

        sum_d += d_loss.item()
        sum_g += g_loss.item()
        n_batches += 1
        step += 1

    avg_d = sum_d / n_batches
    avg_g = sum_g / n_batches
    losses_d.append(avg_d)
    losses_g.append(avg_g)

    print(
        f"Epoch [{epoch+1:2d}/{epochs}]  D: {avg_d:.3f}  G: {avg_g:.3f}  "
        f"a_running: {a_running.item():.3f}"
    )

    G.eval()
    with torch.no_grad():
        sample_imgs = G(fixed_z)
    save_image(
        sample_imgs.detach().cpu(),
        f"samples_StyleGAN3/epoch_{epoch+1:02d}.png",
        nrow=8,
        normalize=True,
    )
    G.train()

print('Training done.')

## 14. Random Samples at 32x32

In [ ]:
G.eval()
with torch.no_grad():
    samples = G(fixed_z)

grid = make_grid(samples.cpu(), nrow=8, normalize=True)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
plt.axis('off')
plt.title('StyleGAN3 — random samples at 32x32')
plt.show()

## 15. Equivariance Demo — Same `z`, Different Coordinate Grids

The conceptual claim is that the synthesis network treats coordinates as continuous. We can probe that claim directly: feed the *same* latent vector through the synthesis network while shifting the coordinate grid, and see whether the output follows the shift.

We approximate the shifted-coordinate path by shifting the Fourier features themselves — that's the only coordinate-dependent input to the synthesis network.

In [ ]:
G.eval()

# Pick one z, run it three times with three different coordinate grids
z_one = torch.randn(1, latent_dim, device=device)

def shift_fourier_features(shift_y=0.0, shift_x=0.0):
    """Override fourier_features with a coordinate-shifted version."""
    H, W = 4, 4
    ys = torch.linspace(-1 + shift_y, 1 + shift_y, H, device=device)
    xs = torch.linspace(-1 + shift_x, 1 + shift_x, W, device=device)
    yy, xx = torch.meshgrid(ys, xs, indexing='ij')
    proj_x = xx.unsqueeze(0) * FOURIER_FREQS[:, 0].unsqueeze(1).unsqueeze(2)
    proj_y = yy.unsqueeze(0) * FOURIER_FREQS[:, 1].unsqueeze(1).unsqueeze(2)
    proj = 2.0 * np.pi * (proj_x + proj_y)
    feats = torch.cat([torch.sin(proj), torch.cos(proj)], dim=0)
    return feats.unsqueeze(0)   # (1, 2*num_freqs, H, W)

with torch.no_grad():
    w_one = G.mapping(z_one)
    img_center = G.synthesis(w_one)

    G.synthesis.input_proj.weight.data  # ensure initialized

    # Replace the input projection call manually for shifted variants
    def img_with_shift(shift_y=0.0, shift_x=0.0):
        feats = shift_fourier_features(shift_y, shift_x)
        x = G.synthesis.input_proj(feats)
        x = G.synthesis.conv_4(x, w_one)
        x = F.leaky_relu(x, 0.2, inplace=True)
        for upsample, conv, blur in zip(G.synthesis.upsamples, G.synthesis.convs, G.synthesis.blurs):
            x = upsample(x)
            x = conv(x, w_one)
            x = F.leaky_relu(x, 0.2, inplace=True)
            x = blur(x)
        return G.synthesis.to_rgb(x)

    img_shift_y = img_with_shift(shift_y=0.4)
    img_shift_x = img_with_shift(shift_x=0.4)

# Stack horizontally
comparison = torch.cat([img_center.cpu(), img_shift_y.cpu(), img_shift_x.cpu()], dim=0)
grid = make_grid(comparison, nrow=1, normalize=True)
plt.figure(figsize=(6, 18))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
plt.axis('off')
plt.title('Top: default coords | Middle: coords shifted +y | Bottom: coords shifted +x')
plt.show()

## 16. Training Progression

In [ ]:
from PIL import Image
import glob

paths = sorted(glob.glob('samples_StyleGAN3/epoch_*.png'))
if paths:
    fig, axes = plt.subplots(1, len(paths), figsize=(1.4 * len(paths), 1.4))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        ax.imshow(np.array(Image.open(p)).squeeze(), cmap='gray')
        ax.set_title(p.split('_')[-1].split('.')[0], fontsize=7)
        ax.axis('off')
    plt.suptitle('StyleGAN3 — same fixed z, sampled every epoch')
    plt.show()
else:
    print('No sample grids found. Run the training cell first.')

## 17. Loss Curves and Path Length

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(range(1, epochs + 1), losses_d, label='D', color='tab:blue')
axes[0].set_xlabel('Epoch')
axes[0].set_title('Discriminator loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(range(1, epochs + 1), losses_g, label='G (adv)', color='tab:orange')
axes[1].set_xlabel('Epoch')
axes[1].set_title('Generator loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

if pl_history:
    axes[2].plot(pl_history, color='tab:green')
    axes[2].set_xlabel('PLR step')
    axes[2].set_title(f'Path length (a_running = {a_running.item():.3f})')
    axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Recap — How StyleGAN3 Differs From StyleGAN2

| Concern | StyleGAN2 | StyleGAN3 |
|---|---|---|
| Initial synthesis input | **learned constant** | **2D Fourier features** |
| Coordinate treatment | opaque (constant absorbs everything) | explicit continuous coordinates |
| After nonlinear ops | no filter | **fixed Gaussian blur** (low-pass) |
| Aliasing control | none | **explicit filtering around every nonlinear stage** |
| Translation equivariance | weak (tied to pixel grid) | stronger (coordinates are the input) |
| Modulated conv | yes | **yes** |
| Demodulation | yes | **yes** |
| Path length regularization | yes | **yes** |
| Loss | adversarial BCE + PLR | **same** |
| Progressive growing | removed | **n/a** |

**What changed and why.**

- **Fourier features replace the learned constant.** StyleGAN2's learned constant was an opaque `(1, C, 4, 4)` tensor. StyleGAN3 explicitly builds its initial feature map from continuous spatial coordinates via sin/cos of random 2D frequencies. The model can't hide coordinate-specific information anywhere — the input is *defined* by coordinates.
- **Low-pass filters after every nonlinearity.** ReLU / LeakyReLU / leaky ReLU can create frequencies that exceed what the next downsampling step can represent. StyleGAN3 applies a fixed Gaussian blur after each nonlinearity to control the frequency content. This is the *core* alias-control mechanism.
- **The signal-processing view.** StyleGAN3 stops thinking of the Generator as a pixel-grid convnet and starts thinking of it as a *signal-processing system* that happens to be sampled at the end. That conceptual shift is what motivates both the Fourier features and the post-nonlinearity blurs.

**Common implementation pitfalls** — quick reference:

- **Fourier frequencies are *frozen*.** They are sampled once at module construction and never updated. If they were learned, the model could still hide pixel-grid-specific information in the input — defeating the point.
- **The blur is *fixed*, not learned.** A learned blur is just an extra conv layer; it doesn't enforce any specific frequency cut-off. Use a known good low-pass filter (Gaussian, with fixed kernel).
- **The blur goes *after* the nonlinearity, not before.** It's the nonlinearity that creates the new high-frequency content. Filtering before the nonlinearity doesn't help.
- **The Fourier features have shape `(B, 2*num_freqs, H, W)`** — twice the number of frequencies because we concatenate sin and cos. Easy to forget the `*2`.
- **Equivariance isn't perfect.** Our simplified architecture is *more* equivariant than StyleGAN2, not perfectly equivariant. The strided convolutions and pixel-aligned upsampling still introduce some grid dependence. The original StyleGAN3 paper uses a more elaborate architecture to get *exact* equivariance.
- **The Discriminator is unchanged.** Alias-control is entirely a Generator-side concern. Reusing the PatchGAN from earlier notebooks is correct.

**Why StyleGAN3 matters.** StyleGAN2 produced clean, controllable images, but the underlying representation was tied to the discrete pixel grid in subtle ways: textures "stuck" to specific locations, small translations of the latent could produce sudden spatial jumps, and the model was hard to use for video or animation. StyleGAN3's signal-processing view addresses all of these by making the coordinate system continuous from the start.

**Next in the series** (Module 21): GAN Evaluation. After 20 modules of building generators, the practical engineering question becomes: *how do you actually measure whether a GAN is good?* The answer isn't "look at the loss curves" — GAN losses are famously uninformative. The module covers Inception Score, Fréchet Inception Distance, precision/recall, and the limits of each.